In [1]:
import sys
sys.path.insert(0, '/app')
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
    
import pytest
import pandas as pd
from pathlib import Path
from typing import List, Optional

from features.base_dataframe import BaseDataFrame
from features.features_utils import FeatureType
from merger.merger_utils import merge_multiple_dataframes_from_parquet
from core.data_org import (
    BUNDLE_DIR,
    get_normalised_instrument_dir,
    MktDataTFreq,
    ExchangeNAME,
    ProductType,
)
from core.enums import g_index_col
from norm.norm_utils import load_normalized_df




etf_x_list = ['QQQ', 'SPY', 'SHV', 'TLT', 'GLD', 'VWO', 'XLB', 'XLE', 'GDX', 'XME', 'XLK', 'XLP', 'XLI', 'VNQ', 'XLV', 'DBC']
etf_x_list_product_type = [ProductType.ETF] * len(etf_x_list)
fx_list = [
'EURUSD',
'USDJPY',
'GBPUSD',
'AUDUSD',
'USDCAD',
'USDCHF',
'NZDUSD',
'EURGBP',
'USDCNH',
'USDHKD']
fx_list_product_type = [ProductType.SPOT] * len(etf_x_list)

asset_list = ['VOO',
'VEA',
'VWO',
'IEF',
'LQD',
'VNQ',
'GLD',
'DBC',
'SCHD',
'QQQ',
'EURUSD',
'USDJPY',
'GBPUSD',
'AUDUSD',
'USDCAD',
'USDCHF',
'NZDUSD',
'EURGBP',
'USDCNH',
'USDHKD']
asset_product_type = [
    ProductType.ETF,ProductType.ETF,ProductType.ETF,ProductType.ETF,ProductType.ETF,
    ProductType.ETF,ProductType.ETF,ProductType.ETF,ProductType.ETF,ProductType.ETF,
    ProductType.SPOT,ProductType.SPOT,ProductType.SPOT,ProductType.SPOT,ProductType.SPOT,
    ProductType.SPOT,ProductType.SPOT,ProductType.SPOT,ProductType.SPOT,ProductType.SPOT
]

DATA_FREQ = MktDataTFreq.CANDLE_1HOUR
SOURCE = ExchangeNAME.FIRSTRATE
PRODUCT_TYPE = ProductType.ETF

FEATURE_TYPES = [
    FeatureType.PRICE,
    FeatureType.RETURN,
    FeatureType.LOG_RETURN,
    FeatureType.VOLUME,
    FeatureType.HIST_VOLATILITY,
    FeatureType.LAG_DELTAS,
    FeatureType.RSI,
    FeatureType.EMA,
    FeatureType.SPREAD_REL_EMA,
    FeatureType.DIFF_REL_EMA_MID,
    FeatureType.ADI,
    FeatureType.ROC,
]

DATA_FREQ = MktDataTFreq.CANDLE_1HOUR
ASSET_LIST = asset_list

def get_base_df(freq = DATA_FREQ, source = SOURCE, product_type= PRODUCT_TYPE, symbol = 'QQQ'):

    instrument_dir = get_normalised_instrument_dir(freq, source, product_type, symbol)

    parquet_files = list(instrument_dir.glob("*.df.parquet"))
    if not parquet_files:
        raise FileNotFoundError(f"No parquet files found in {instrument_dir}")

    input_file = parquet_files[0]
    print(f"Loading: {input_file}")

    df = load_normalized_df(str(input_file))
    base_df = BaseDataFrame(
        p_df=df,
        p_valid_col_name="valid_row",
        p_scaling=-1,
        p_verbose=False,
    )
    return base_df

In [3]:
# this load a list of df
assets_list_dfs = []
for i in range(len(asset_list)):
    assets_list_dfs.append(get_base_df(product_type=asset_product_type[i], symbol = asset_list[i]))

Loading: /app/data/normalised/candle_1hour/firstrate_undefined/etf/VOO/VOO_*_*_candle_1hour.df.parquet
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/etf/VEA/VEA_*_*_candle_1hour.df.parquet
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/etf/VWO/VWO_*_*_candle_1hour.df.parquet
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/etf/IEF/IEF_*_*_candle_1hour.df.parquet
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/etf/LQD/LQD_*_*_candle_1hour.df.parquet
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/etf/VNQ/VNQ_*_*_candle_1hour.df.parquet
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/etf/GLD/GLD_*_*_candle_1hour.df.parquet
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/etf/DBC/DBC_*_*_candle_1hour.df.parquet
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/etf/SCHD/SCHD_*_*_candle_1hour.df.parquet
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/etf/QQQ/

In [5]:
"""
Unit tests for ETF feature bundle computation.

Tests building features for ETFs and merging them into a bundle using the fwk modules.
"""

def compute_features_for_asset(freq:MktDataTFreq = DATA_FREQ, source:ExchangeNAME = SOURCE, product_type:ProductType = PRODUCT_TYPE , symbol: str = 'QQQ', features : bool = False) -> Path:
    """
    Compute features for a single Asset and save to bundle directory.

    Returns:
        Path to the saved feature parquet file
    """
    instrument_dir = get_normalised_instrument_dir(freq, source, product_type, symbol)

    parquet_files = list(instrument_dir.glob("*.df.parquet"))
    if not parquet_files:
        raise FileNotFoundError(f"No parquet files found in {instrument_dir}")

    input_file = parquet_files[0]
    print(f"Loading: {input_file}")

    df = load_normalized_df(str(input_file))
    print(f"  Loaded {df.shape[0]} rows, {df.shape[1]} columns")

    base_df = BaseDataFrame(
        p_df=df,
        p_valid_col_name="valid_row",
        p_scaling=-1,
        p_verbose=False,
    )

    if features:
        for feature_type in FEATURE_TYPES:
            base_df.add_feature(feature_type)

    base_df.convert_f16_columns()

    result_df = base_df.get_dataframe()
    feature_cols = base_df.get_feature_columns()

    print(f"  Result: {result_df.shape[0]} rows, {len(feature_cols)} features")

    output_dir = BUNDLE_DIR / DATA_FREQ.value
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / f"{symbol}_features.parquet"

    result_df.to_parquet(output_file, index=False)
    print(f"  Saved: {output_file}")

    return output_file


def compute_asset_bundle(
    asset_product_type: Optional[List[str]] = None,
    asset_list: Optional[List[str]] = None,
    output_prefix: str = "asset_bundle",
    output_dir: Path = BUNDLE_DIR,
    features: bool = False
) -> Path:
   
    if asset_product_type is None:
        return null
    if asset_list is None:
        return null
    if output_dir is None:
        output_dir = BUNDLE_DIR

    output_dir.mkdir(parents=True, exist_ok=True)

    feature_files = []
    for i in range(len(asset_list)):
        print(f"\n{'=' * 60}")
        print(f"Processing: {asset_list[i]}")
        print(f"{'=' * 60}")
        feature_file = compute_features_for_asset(product_type = asset_product_type[i],symbol = asset_list[i], features = features)
        feature_files.append(feature_file)

    print(f"\n{'=' * 60}")
    print("Merging All feature files...")
    print(f"{'=' * 60}")

    merged_df = merge_multiple_dataframes_from_parquet(
        file_paths=[str(f) for f in feature_files],
        p_names=asset_list,
        p_cols_list=[],
        p_id_col=g_index_col,
        p_float_16=False,
    )

    print(f"\nMerged dataframe:")
    print(f"  Shape: {merged_df.shape}")

    feature_cols = [c for c in merged_df.columns if "_F_" in c]
    print(f"  Total feature columns: {len(feature_cols)}")

    output_path = output_dir / f"{output_prefix}_features_bundle.parquet"
    merged_df.to_parquet(output_path)

    file_size_mb = output_path.stat().st_size / (1024 * 1024)
    print(f"\n{'=' * 60}")
    print(f"Bundle saved to: {output_path}")
    print(f"File size: {file_size_mb:.2f} MB")
    print(f"{'=' * 60}")

    return output_path

In [7]:
output_file = compute_asset_bundle(
    asset_product_type = asset_product_type,
    asset_list = asset_list ,
    output_prefix="asset_list_",
    output_dir=BUNDLE_DIR,
)

print(f"\n{'=' * 60}")
print(f"Bundle file: {output_file}")
print(f"{'=' * 60}")

assert output_file.exists(), f"Bundle file should exist: {output_file}"


Processing: VOO
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/etf/VOO/VOO_*_*_candle_1hour.df.parquet
  Loaded 47171 rows, 8 columns
  Result: 47171 rows, 4 features
  Saved: /app/data/bundle/candle_1hour/VOO_features.parquet

Processing: VEA
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/etf/VEA/VEA_*_*_candle_1hour.df.parquet
  Loaded 41306 rows, 8 columns
  Result: 41306 rows, 4 features
  Saved: /app/data/bundle/candle_1hour/VEA_features.parquet

Processing: VWO
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/etf/VWO/VWO_*_*_candle_1hour.df.parquet
  Loaded 49451 rows, 8 columns
  Result: 49451 rows, 4 features
  Saved: /app/data/bundle/candle_1hour/VWO_features.parquet

Processing: IEF
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/etf/IEF/IEF_*_*_candle_1hour.df.parquet
  Loaded 53125 rows, 8 columns
  Result: 53125 rows, 4 features
  Saved: /app/data/bundle/candle_1hour/IEF_features.parquet

Processing: LQD
Loading